# 第43章　セグメンテーション結果の定量化と後処理

**『医療診断支援AI開発　第2巻　実装編 ― 本格実装から臨床導入まで（第2巻　実装編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-impl

## 連結成分と後処理を、コードで

In [ ]:
import numpy as np
from scipy import ndimage

def quantify(mask, spacing):               # spacing=(sz, sy, sx) [mm]
    sp = np.asarray(spacing, dtype=float)
    if sp.shape != (3,) or not np.all(np.isfinite(sp)) or np.any(sp <= 0):
        return None                        # 不正な spacing は評価不能（空の結果＝病変なし、と区別する）
    voxel_vol = float(np.prod(sp))         # 1ボクセルの体積[mm^3]
    labels, n = ndimage.label(mask)        # 連結成分に分割
    results = []
    for i in range(1, n + 1):
        blob = labels == i
        vox = blob.sum()
        vol = vox * voxel_vol
        if vol < MIN_LESION_VOL:           # 臨床的下限より小さい塊は除外
            continue
        # 体積から求めた等価球径。3次元の最大径とも、臨床の最長径（軸位断）とも別物なので、
        # キー名で区別する。同じ "diameter_mm" に入れると、下流で必ず取り違える。
        results.append({"volume_mm3": vol,
                        "equivalent_sphere_diameter_mm":
                            2 * (3 * vol / (4 * np.pi)) ** (1/3)})
    return results

## マスクの先の情報 ― ラジオミクス（radiomics）

In [ ]:
import SimpleITK as sitk
from radiomics import featureextractor
extractor = featureextractor.RadiomicsFeatureExtractor()
extractor.settings["binWidth"] = 25          # 画素値（CT値）の量子化幅（再現性に直結）
feats = extractor.execute(sitk.ReadImage(ct_path),
                          sitk.ReadImage(mask_path))   # 数百特徴を一括抽出